In [0]:
%sql
CREATE OR REPLACE TABLE cpt_utility_catalog.gold.fct_cpi
AS
SELECT
xxhash64(c.id) AS cpi_key,

COALESCE(CAST(date_format(c.date,'yyyyMMdd') AS INT), -1) AS date_key,
COALESCE(dp.product_key, xxhash64('unmapped')) AS product_key,
COALESCE(dg.geography_key, xxhash64('unmapped')) AS geography_key,

c.index_value AS index_value,
decile AS decile


FROM cpt_utility_catalog.silver.silver_cpi_cleaned c

LEFT JOIN cpt_utility_catalog.gold.dim_product dp
ON xxhash64(concat_ws('||', 
            LOWER(TRIM(c.series_identifier)), 
            LOWER(TRIM(c.category)), 
            LOWER(TRIM(c.subcategory)), 
            LOWER(TRIM(c.survey_code))
        )) = dp.product_key

LEFT JOIN cpt_utility_catalog.gold.dim_geonational dg
ON xxhash64(LOWER(TRIM(c.geographic_area))) = dg.geography_key

